# Lesson 12: Time Series Operations
**Module 2: Data Manipulation**

Interactive notebook for practicing time series operations.

## Learning Objectives
- Work with DatetimeIndex
- Resample time series data
- Use shift() and rolling windows
- Calculate returns and technical indicators

In [ ]:
import pandas as pd
import numpy as np
np.random.seed(42)

# Create sample stock data with dates
dates = pd.date_range('2024-01-01', periods=100, freq='B')  # Business days
df = pd.DataFrame({
    'Price': 100 + np.random.randn(100).cumsum() * 2,
    'Volume': np.random.randint(1000000, 5000000, 100)
}, index=dates)

print('Sample data:')
print(df.head())
print(f'\nIndex type: {type(df.index)}')

## 1. DatetimeIndex Operations
Powerful date-based slicing and attributes.

In [ ]:
# Date slicing
print('January 2024 data:')
print(df['2024-01'].head())

print('\nFirst week of January:')
print(df['2024-01-01':'2024-01-05'])

In [ ]:
# Date attributes
df['DayOfWeek'] = df.index.dayofweek
df['Month'] = df.index.month
df['Quarter'] = df.index.quarter
print(df[['Price', 'DayOfWeek', 'Month', 'Quarter']].head())

## 2. Resampling
Change time series frequency.

In [ ]:
# Resample to weekly
weekly = df['Price'].resample('W').agg(['first', 'max', 'min', 'last'])
weekly.columns = ['Open', 'High', 'Low', 'Close']
print('Weekly OHLC:')
print(weekly.head())

In [ ]:
# Resample to monthly
monthly = df['Price'].resample('M').mean()
print('Monthly average prices:')
print(monthly)

## 3. Shift and Returns
Calculate returns using shift().

In [ ]:
# Simple returns
df['Return'] = df['Price'].pct_change()

# Log returns
df['LogReturn'] = np.log(df['Price'] / df['Price'].shift(1))

print('Returns:')
print(df[['Price', 'Return', 'LogReturn']].head(10))

In [ ]:
# Return statistics
print(f'Mean daily return: {df["Return"].mean():.4%}')
print(f'Daily volatility: {df["Return"].std():.4%}')
print(f'Annualized volatility: {df["Return"].std() * np.sqrt(252):.2%}')

## 4. Rolling Windows
Calculate moving statistics.

In [ ]:
# Moving averages
df['MA5'] = df['Price'].rolling(5).mean()
df['MA20'] = df['Price'].rolling(20).mean()

print('Moving averages:')
print(df[['Price', 'MA5', 'MA20']].tail(10))

In [ ]:
# Rolling volatility
df['Volatility20'] = df['Return'].rolling(20).std() * np.sqrt(252)

print('Rolling volatility:')
print(df[['Return', 'Volatility20']].dropna().tail())

## 5. Cumulative Returns

In [ ]:
# Cumulative returns
df['CumReturn'] = (1 + df['Return']).cumprod() - 1

print('Cumulative returns:')
print(f'Total return: {df["CumReturn"].iloc[-1]:.2%}')
print(df[['Price', 'Return', 'CumReturn']].tail())

## Exercise: Create Trading Signals
Build a simple moving average crossover strategy.

In [ ]:
# Exercise: Create trading signals
# 1. Calculate 5-day and 20-day moving averages (already done above)
# 2. Create 'Signal': 1 when MA5 > MA20 (bullish), 0 otherwise
# 3. Create 'Position': Signal shifted by 1 (trade next day)
# 4. Calculate strategy return: Position * Return

# YOUR CODE HERE

---
## Solution
<details>
<summary>Click to reveal</summary>

```python
# Create signal
df['Signal'] = (df['MA5'] > df['MA20']).astype(int)

# Position with lag (trade next day)
df['Position'] = df['Signal'].shift(1)

# Strategy returns
df['StrategyReturn'] = df['Position'] * df['Return']
df['CumStrategyReturn'] = (1 + df['StrategyReturn']).cumprod() - 1

# Compare performance
print(f'Buy & Hold Return: {df["CumReturn"].iloc[-1]:.2%}')
print(f'Strategy Return: {df["CumStrategyReturn"].iloc[-1]:.2%}')
print(f'\nSignal distribution:')
print(df['Signal'].value_counts())
```
</details>